In [1]:
# %pip install -q langchain-ollama

In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="deepseek-r1:1.5b")  
llm.invoke("What is the capital of France?") # 프롬프트 : LLM호출 

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-03-13T01:47:23.0300745Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4879039400, 'load_duration': 4762569600, 'prompt_eval_count': 10, 'prompt_eval_duration': 19477300, 'eval_count': 12, 'eval_duration': 89834300, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--019ce4e0-6903-7c03-a78c-e34dfb64d448-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 12, 'total_tokens': 22})

In [2]:
# ValueError: Invalid input type <class 'int'>. Must be a PromptValue, str, or list of BaseMessages.
# -> invoke 할때 나름의 입력형식이 정해져 있다. 

In [3]:
# PromptValue -> PRomptTemplate을 활용하면 만들 수 있다

In [4]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    template = "What is the capital of {country}?",  # {country} : placeholder -> 대입을 해줘야 한다. HOW?
    input_variables=["country"],
)

## LLM에 invoke 한 것처럼, 프롬프트도 invoke를 해줘야 한다 
prompt = prompt_template.invoke({"country":"France"})  # prompt 가 promptValue가 됨
print(prompt)  # output : text='What is the capital of France?'

text='What is the capital of France?'


In [5]:
llm.invoke(prompt)  # 잘 들어간다 -> 질문은 France로 대입 된 것
# 즉, invoke가 2번 들어가게 되는 것 (프롬프트 1, LLM 1)

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-03-13T01:47:23.2732671Z', 'done': True, 'done_reason': 'stop', 'total_duration': 201232300, 'load_duration': 90983200, 'prompt_eval_count': 10, 'prompt_eval_duration': 8707600, 'eval_count': 12, 'eval_duration': 89566200, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--019ce4e0-7c3f-7df2-b8d1-f85c5b3836cb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 12, 'total_tokens': 22})

#### BaseMessageList

In [6]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# llm.invoke([HumanMessage(content="What is the capital of {country}?")]) # 리스트 형식으로 넣을 것

# HumanMessage를 가장 마지막에 넣어서 invoke -> 답변 생성됨
# Few-Shot : 예제를 제공; 답변의 형식에 대한 예제. -> 마치 대화 이력이 있는 것처럼 LLM을 속임 -> 답변을 원하는대로 유도하는 기술
# Few-Shoe 논문 참고
# 복잡한 어플리케이션을 구현할 때는 예제를 주는게 정확도가 좋다
message_list = [
    SystemMessage(content="You are a helpful assistant"),
    HumanMessage(content="What is the capital of France?"),
    AIMessage(content="The capital of France is Paris."),
    HumanMessage(content="What is the capital of Germany?"),
    AIMessage(content="The capital of France is Berlin."),
    HumanMessage(content="What is the capital of Italy?"),
    AIMessage(content="The capital of France is Rome."),
    HumanMessage(content="What is the capital of {country}?"),
]


llm.invoke(message_list)


AIMessage(content='Hi there! I just noticed something about your question that I need to clarify. It says, "What is the capital of {country}?" but it doesn\'t specify which country you\'re referring to. Could you please provide more details or let me know which country you\'re interested in? I\'d be happy to help with that information!', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-03-13T01:47:24.1236205Z', 'done': True, 'done_reason': 'stop', 'total_duration': 831473300, 'load_duration': 86914400, 'prompt_eval_count': 67, 'prompt_eval_duration': 22817500, 'eval_count': 73, 'eval_duration': 628627800, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--019ce4e0-7d1b-71f0-a1eb-3716ff751d50-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 67, 'output_tokens': 73, 'total_tokens': 140})

- 위와 같은 방법은 Langchain 스럽지 않은 방법 -> ChatPromptTemplate을 활용하자
- ChatPromptTemplate : 채팅 메시지처럼 사용하는 방법이 있음

In [7]:
from langchain_core.prompts import ChatPromptTemplate

# ### 잘못된 방법 -> 아래 셀을 참고하자
# chat_prompt_template = ChatPromptTemplate.from_messages(message_list)  # 이 부분을 수정!
# chat_prompt = chat_prompt_template.invoke({"country" : "France"})   
# print(chat_prompt.messages)   # HumanMessage(content='What is the capital of {country}? -> placeholder에 안들어가있다

In [10]:

chat_prompt_template = ChatPromptTemplate.from_messages([
    # 튜플 형식으로 넣어줘야 한다 - few shot 도 이런 형식으로 활용할 것 (위 방식보다는 LCEL을 구축하는데에 있어 확장성에서도 유리)
    ("system", "You are a helpful assistant"),
    ("human", "What is the capital of {country}?"),
])
chat_prompt = chat_prompt_template.invoke({"country" : "France"})   
print(chat_prompt.messages)   # HumanMessage(content='What is the capital of France? -> 잘 대입됨

[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={})]


In [11]:
llm.invoke(chat_prompt)  # 위 셀처럼 해야 LLM 답변도 잘 나온다

AIMessage(content='The capital of France is Paris. This city is also the capital of the Seine River region and has been the capital since 1960. As of 2023, Paris remains the capital under the Paris Commune government until it was divided in 2012.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-03-13T01:50:01.7915661Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3929495500, 'load_duration': 95754000, 'prompt_eval_count': 15, 'prompt_eval_duration': 150126000, 'eval_count': 402, 'eval_duration': 3104157400, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--019ce4e2-d8e4-7ec3-ad52-c6206ccc139c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 402, 'total_tokens': 417})